Load data from csv locate in data/raw

Historical Dublin Bikes data are provided under multiple schema versions. We standardised all datasets to a common GBFS-inspired schema prior to analysis.

General Bikeshare Feed Specificationm (GBFS).
It is a standard data schema defined by the North American Bikeshare Association (NABSA) for publishing bike-sharing system data in a consistent way.

In [1]:

import pandas as pd
from pathlib import Path

# GBFS columns we keep

FINAL_COLUMNS = [
    "system_id",
    "last_reported",
    "station_id",
    "num_bikes_available",
    "num_docks_available",
    "is_installed",
    "is_renting",
    "is_returning",
    "name",
    "address",
    "lat",
    "lon",
    "capacity",
]

DROP_COLUMNS = [
    "system_id",
    "is_installed",
    "is_renting",
    "is_returning",
]

# Column mappings for old files

OLD_TO_NEW = {
    "STATION ID": "station_id",
    "TIME": "last_reported",
    # "LAST UPDATED": "last_reported", --- IGNORE ---   
    "NAME": "name",
    "ADDRESS": "address",
    "LATITUDE": "lat",
    "LONGITUDE": "lon",
    "BIKE_STANDS": "capacity",
    "AVAILABLE_BIKES": "num_bikes_available",
    "AVAILABLE_BIKE_STANDS": "num_docks_available",
}


def load_and_standardize_csv(path: Path) -> pd.DataFrame:
    """
    Load a Dublin Bikes CSV and standardise it to the GBFS schema.
    """
    df = pd.read_csv(path)

    # Normalize column names (strip whitespace)
    # df.columns = df.columns.str.strip()

    # Case 1: old-format file
    if "STATION ID" in df.columns:
        df = df.rename(columns=OLD_TO_NEW)

        # Add missing GBFS-style columns
        df["system_id"] = "dublin_bikes"
        df["is_installed"] = True
        df["is_renting"] = True
        df["is_returning"] = True

    # Case 2: newer-format file
    else:
        # Drop columns we do not need if present
        df = df.copy()

    # Keep only columns we care about (ignore extras)
    df = df[[c for c in FINAL_COLUMNS if c in df.columns]]

    # Enforce column order
    df = df.reindex(columns=FINAL_COLUMNS)

    return df


def load_all_csvs(data_dir: Path) -> pd.DataFrame:
    """
    Load and concatenate all CSV files in a directory.
    """
    dfs = []

    for csv_file in sorted(data_dir.glob("*.csv")):
        print(f"Loading {csv_file.name}")
        dfs.append(load_and_standardize_csv(csv_file))

    return pd.concat(dfs, ignore_index=True)


def save_dataframe(df: pd.DataFrame, output_path: Path) -> None:
    """
    Save the consolidated DataFrame to CSV.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Saved processed data to {output_path}")



if __name__ == "__main__":
    raw_data_path = Path("../data/raw")
    output_file = Path("../data/processed/dublin_bikes_all.csv")


    df_all = load_all_csvs(raw_data_path)

    # Drop non-informative columns
    df_all = df_all.drop(columns=DROP_COLUMNS, errors="ignore")

    # Save processed dataset
    save_dataframe(df_all, output_file)

    print(df_all.head())
    print(df_all.info())





Loading dublinbike-historical-data-2022-01.csv
Loading dublinbike-historical-data-2022-02.csv
Loading dublinbike-historical-data-2022-03.csv
Loading dublinbike-historical-data-2022-04.csv
Loading dublinbike-historical-data-2022-05.csv
Loading dublinbike-historical-data-2022-06.csv
Loading dublinbike-historical-data-2022-07.csv
Loading dublinbike-historical-data-2022-08.csv
Loading dublinbike-historical-data-2022-09.csv
Loading dublinbike-historical-data-2022-10.csv
Loading dublinbike-historical-data-2022-11.csv
Loading dublinbike-historical-data-2022-12.csv
Loading dublinbike-historical-data-2023-01.csv
Loading dublinbike-historical-data-2023-02.csv
Loading dublinbike-historical-data-2023-03.csv
Loading dublinbike-historical-data-2023-04.csv
Loading dublinbike-historical-data-2023-05.csv
Loading dublinbike-historical-data-2023-06.csv
Loading dublinbike-historical-data-2023-07.csv
Loading dublinbike-historical-data-2023-08.csv
Loading dublinbike-historical-data-2023-09.csv
Loading dubli